# Set Up

In [431]:
#Import libraries
import pandas as pd
import numpy as np
import re, os, sqlite3
from datetime import datetime

### Read in Data

In [432]:
input_path = '/Users/minji.kang/Documents/NGDT/Data_export_management/Report_CSV_Preprocessing_Generic_Script/TestLookUp/input/'
output_path = '/Users/minji.kang/Documents/NGDT/Data_export_management/Report_CSV_Preprocessing_Generic_Script/TestLookUp/output/'

def read_and_bind_df(mypath) :
    all_files = os.listdir(mypath)
    report_files = [file for file in all_files if file.startswith('report')]
    report_df = []
    for i in range(len(report_files)):
        temp_df = pd.read_csv(os.path.join(mypath, report_files[i]), encoding='ISO-8859-1')
        report_df.append(temp_df)
    report = pd.concat(report_df, ignore_index=True)
    report.rename(columns={report.columns[0]: 'id'}, inplace=True) 
    return report

df = read_and_bind_df(input_path)
df.to_csv(os.path.join(output_path,'report_all.csv'),index=False)


### Response History File Creation

In [433]:
def response_history(data):
    copy_data = data.copy()
    copy_data = copy_data[['version', 'activity_flow_id', 'activity_flow_name', 'activity_id', 'activity_name', 'item_id', 'item', 'prompt', 'options' ]].drop_duplicates()
    return copy_data

response_history_df = response_history(df)
response_history_df.to_csv(os.path.join(output_path,'response_history.csv'),index=False)


In [435]:
def extract_scores(data):
    df_scores = data.iloc[:, list(range(24)) + [29, 31, 32, 33, 34] + list(range(35, data.shape[1]))].drop_duplicates()
    df_long = df_scores.melt(id_vars=df_scores.columns[:29].tolist(),  # Keep the first column as identifier
                            var_name="item",         # Name of the new column for column names
                            value_name="response")  
    df_long = df_long.dropna(subset=['response'])
    score_list = df_long['item'].unique()
    lookup_names = [x for x in score_list if re.match(r'^Optional text for ', x) and x != 'Optional text for Final SubScale Score']
    lookup_names = [x.replace('Optional text for ', '') for x in lookup_names]
    df_long['score_type'] = np.where(df_long['item'] == 'Final SubScale Score', 'finalscore',
                                     np.where(df_long['item'] == 'Optional text for Final SubScale Score', 'finalscore_text',
                                              np.where(df_long['item'].isin(lookup_names), 'lookup',
                                                       np.where(df_long['item'].str.contains(r'^Optional text for ', regex=True), 'lookup_text', 'subscale'))))
    df_long['item'] = df_long['item'].str.replace(r'\s+', '_', regex=True)
    df_long['item'] = np.where(df_long['score_type']=='finalscore', 'activity_score', 
                               np.where(df_long['score_type']=='finalscore_text', 'activity_score_lookup_text', 
                                        np.where(df_long['score_type']=='lookup', 'subscale_lookup_' + df_long['item'], 
                                                 np.where(df_long['score_type']=='lookup_text', 'subscale_lookup_text_' + df_long['item'].str.replace(r'^Optional_text_for_', '', regex=True), 
                                                          'subscale_name_' + df_long['item']))))
    test = df_long.copy()
    df_long.drop('score_type', axis=1, inplace=True)
    df_subset = data.iloc[:, list(range(35))]
    df_column_list = df_subset.columns
    df_long['item_id'] = 'no id'
    df_long['prompt'] = 'prompt_' + df_long['item'] 
    df_long['options'] = 'options_' + df_long['item']
    df_long['rawScore'] = 'rawScore_' + df_long['item']
    df_column_list2 = df_long.columns
    # print(df_column_list2)
    df_long = df_long[df_column_list.tolist()]
    df_output = pd.concat([df_subset, df_long], axis=0)
    return df_output.reset_index()
    # return test

df = extract_scores(df)

df.to_csv(os.path.join(output_path,'extracted_scores.csv'),index=False)



### Add Timezone offset

In [436]:
def add_timezone_offset(mydata, columntoaddto):
    col_values = pd.to_numeric(mydata[columntoaddto], errors='coerce')
    timezone_offsets = pd.to_numeric(mydata['timezone_offset'], errors='coerce')
    timezone_offsets = timezone_offsets.fillna(0)  # Replace NaN with 0 for offsets
    return col_values + (timezone_offsets * 60 * 1000)


df['start_Time'] = add_timezone_offset(df, 'activity_start_time')
df['end_Time'] = add_timezone_offset(df, 'activity_end_time')  
df['schedule_Time'] = add_timezone_offset(df, 'activity_scheduled_time')  

# df[['activity_start_time', 'start_Time', 'activity_end_time', 'end_Time', 'activity_scheduled_time', 'schedule_Time', 'timezone_offset']]

### Group by min start_Time & max End_Time

In [437]:
# dat_processed = df.groupby(['secret_user_id', 'activity_flow_id', 'activity_scheduled_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)

### Score options replacements and removing unnecessary characters

In [438]:
def val_score_mapping(data):
    response_scores = []  # List to store results

    for i in range(len(data['response'])):
        options = data['options'][i]
        response = data['response'][i]
        # clean_response = re.sub(r"value: |geo: ", "", response)
        if isinstance(response, str):
            clean_response = re.sub(r"value: |geo: ", "", response)
        else:
            clean_response = np.nan

        # Ensure 'options' and 'response' are valid strings
        if not isinstance(options, str) or not isinstance(response, str):
            response_scores.append(clean_response)  # Append NaN for invalid rows
            continue

        if re.search(r'score: ', options):
            split_options = options.strip().split("),")
            split_response = response.strip().split(": ")[1].split(',')
            scores = {}

            for j in split_options:
                if "(score" in j:  # Ensure the string contains the expected structure
                    val_parts = j.split("(score")
                    if len(val_parts) == 2 and ": " in val_parts[0]:
                        val_num = val_parts[0].split(": ")[1].strip()
                        score_num = val_parts[1].split(": ")[1].strip(" )")
                        scores[val_num] = score_num

            response_score_mapping = {
                resp.strip(): scores.get(resp.strip(), "N/A")  # Handle missing mappings
                for resp in split_response
            }
            response_scores.append(', '.join(response_score_mapping.values()))
        else:
            response_scores.append(clean_response)  # Append NaN if no valid scores are found

    return pd.Series(response_scores)

# df['response_scores'] = val_score_mapping(df)

### Formatting time and time_range items

In [439]:
def format_responses(data):
    formatted_responses = []  # List to store the formatted responses

    for i in range(len(data)):
        response = data['response'].iloc[i]
        
        # Ensure response is a string, or handle NaN/invalid values
        if not isinstance(response, str):
            formatted_responses.append(response)  # Leave as is for non-string values
            continue
        
        # Handle 'time:' entries
        if re.search(r'time:', response):
            if re.search(r'hr [0-9],', response):  # Single-digit hour
                egapp = response.replace('time: hr ', '0')
                if re.search(r', min [0-9]$', egapp):  # Single-digit minute
                    egtemp = egapp.replace(', min ', ':0')
                elif re.search(r', min [0-9][0-9]$', egapp):  # Two-digit minute
                    egtemp = egapp.replace(', min ', ':')
            elif re.search(r'hr [0-9][0-9],', response):  # Two-digit hour
                egapp = response.replace('time: hr ', '')
                if re.search(r', min [0-9]$', egapp):  # Single-digit minute
                    egtemp = egapp.replace(', min ', ':0')
                elif re.search(r', min [0-9][0-9]$', egapp):  # Two-digit minute
                    egtemp = egapp.replace(', min ', ':')
            
            # Convert to formatted time
            egpos = datetime.strptime(egtemp, '%H:%M')
            formatted_responses.append(egpos.strftime('%H:%M'))
        
        # Handle 'time_range:' entries
        elif re.search(r'time_range:', response):
            # Extract times and format them
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', response)  # Remove unwanted characters
            t = t.replace(',', ':')  # Replace commas with colons
            time_parts = t.split('/')  # Split the time range into two parts
            
            # Format each time part
            formatted_parts = []
            for part in time_parts:
                hours, minutes = part.split(':')
                hours = hours.zfill(2)  # Ensure hours are two digits
                minutes = minutes.zfill(2)  # Ensure minutes are two digits
                formatted_parts.append(f"{hours}:{minutes}")
            
            # Combine the formatted parts back into the time range
            formatted_responses.append('/'.join(formatted_parts))
        
        # Handle other cases
        else:
            formatted_responses.append(response)  # Keep the response unchanged

    return pd.Series(formatted_responses)  # Return as a pandas Series

# df['formatted_responses'] = format_responses(df)

### Converting epoch time to regular time

In [440]:
def format_epochtime(data, column_name):
    epoch_converted = []
    epoch_converted = pd.to_numeric(data[column_name], errors='coerce')
    epoch_converted = pd.to_datetime(epoch_converted / 1000, unit='s')

    return epoch_converted

df['start_Time'] = format_epochtime(df, 'start_Time')
df['end_Time'] = format_epochtime(df, 'end_Time')
df['schedule_Time'] = format_epochtime(df, 'schedule_Time')

# df[['activity_start_time', 'start_Time', 'activity_end_time', 'end_Time', 'activity_scheduled_time', 'schedule_Time', 'timezone_offset']]

/Users/minji.kang/anaconda3/lib/python3.11/site-packages/pandas/core/tools/datetimes.py:557: RuntimeWarning: invalid value encountered in cast
  arr, tz_parsed = tslib.array_with_unit_to_datetime(arg, unit, errors=errors)


### Processing Responses Function (combined score mapping+cleaning and formatting)

In [441]:
def process_responses(data, clean=True, map=True, format=True):
    processed_responses = []  # List to store the processed responses

    for i in range(len(data)):
        options = data['options'].iloc[i] if 'options' in data else None
        response = data['response'].iloc[i]

        processed = False 

        # Ensure response is a string or NaN
        if not isinstance(response, str):
            response = str(response) if not pd.isna(response) else np.nan

        # Clean the response
        if clean:
            if isinstance(response, str) and re.search(r'geo:', response):
                geo_match = re.search(r'geo:\s*lat\s*\((.*?)\)\s*/\s*long\s*\((.*?)\)', response)
                if geo_match:
                    lat = geo_match.group(1).strip()
                    long = geo_match.group(2).strip()
                    geo_cleaned = f"{lat}/{long}"
                    processed_responses.append(geo_cleaned)
                    processed = True
                    continue

            if isinstance(response, str) and re.search(r'value:', response):
                clean_response = re.sub(r"value:\s*", "", response)
                processed_responses.append(clean_response)
                processed = True
                continue

            if pd.isna(response):  # Handle NaN explicitly
                processed_responses.append(np.nan)
                processed = True
                continue

        # Format time or time range entries
        if format:
            if isinstance(response, str) and re.search(r'time:', response):
                try:
                    time_match = re.search(r'hr\s*(\d{1,2}),\s*min\s*(\d{1,2})', response)
                    if time_match:
                        hour = int(time_match.group(1))
                        minute = int(time_match.group(2))
                        formatted_time = f"{hour:02}:{minute:02}"
                        processed_responses.append(formatted_time)
                        processed = True
                    else:
                        processed_responses.append(np.nan)
                        processed = True
                except Exception:
                    processed_responses.append(np.nan)
                    processed = True

            elif isinstance(response, str) and re.search(r'time_range:', response):
                try:
                    clean_time = re.sub(r'[a-zA-Z\s+(\)_:]', '', response).replace(',', ':')
                    time_parts = clean_time.split('/')
                    formatted_parts = [
                        f"{part.split(':')[0].zfill(2)}:{part.split(':')[1].zfill(2)}"
                        for part in time_parts
                    ]
                    processed_responses.append('/'.join(formatted_parts))
                    processed = True
                except Exception:
                    processed_responses.append(np.nan)
                    processed = True
                    continue

        # Map scores
        if map and isinstance(response, str) and isinstance(options, str):  
            if re.search(r'score: ', options):
                split_options = options.strip().split("),")
                split_response = response.strip().split(": ")[1].split(',')
                scores = {}

                for j in split_options:
                    if "(score" in j:
                        val_parts = j.split("(score")
                        if len(val_parts) == 2 and ": " in val_parts[0]:
                            val_num = val_parts[0].split(": ")[1].strip()
                            score_num = val_parts[1].split(": ")[1].strip(" )")
                            scores[val_num] = score_num

                response_score_mapping = {
                    resp.strip(): scores.get(resp.strip(), "N/A")
                    for resp in split_response
                }
                processed_responses.append(', '.join(response_score_mapping.values()))
                processed = True
                continue
        
        # Fallback case
        if not processed:
            processed_responses.append(response)

    return pd.Series(processed_responses)



# dat_processed = df.copy()
# dat_processed['new_responses'] = process_responses(df)



### Widening Function

In [442]:
dat_processed = df.copy()
dat_processed['new_responses'] = process_responses(df)

mycolumn_list = ['userId', 'secret_user_id', 
                 'source_user_secret_id',  #'source_user_nickname', 'source_user_tag', 'source_user_relation',   
                 'target_user_secret_id', #'target_user_nickname', 'target_user_tag',
                 'input_user_secret_id', #'input_user_nickname', 
                 'schedule_Time', 'start_Time', 'end_Time',
                 'activity_flow_id', 'activity_flow_name', 
                 'activity_id', 'activity_name',
                 'event_id', 'version' ]
myresponse_column_name = 'new_responses'

def widen_data(data, column_list, response_column_name):
    datetime_cols = data.select_dtypes(include=['datetime']).columns
    data[datetime_cols] = data[datetime_cols].astype(str).replace('NaT', '')
    data[column_list] = data[column_list].fillna('')
    answers = data.groupby(column_list)['id'].apply(lambda x: '|'.join(x.astype(str))).reset_index()
    data['combined_cols'] = data[['item', 'item_id']].astype(str).agg('_(id:'.join, axis=1)
    data['combined_cols'] += ')'
    data['combined_cols'] = np.where(data['combined_cols'].str.contains('no id', regex=False), data['combined_cols'].str.replace('_(id:no id)', '', regex=False), data['combined_cols'])
    subset_columns = column_list + ['combined_cols', response_column_name]
    dat_subset = data[subset_columns]
    dat_wide = pd.pivot_table(dat_subset, index=column_list, columns='combined_cols', values=response_column_name, aggfunc='last').reset_index()
    dat_wide = pd.merge(dat_wide, answers, on=column_list, how='outer')
    return dat_wide

data_wide = widen_data(dat_processed, mycolumn_list, myresponse_column_name)
data_wide.to_csv(os.path.join(output_path,'data_wide_all.csv'),index=False)

# data_wide.head()
